In [13]:
# First let's start with some simple imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from wrf import uvmet, to_np, getvar, interplevel, smooth2d, get_cartopy, cartopy_xlim, cartopy_ylim, latlon_coords,ll_to_xy
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
import matplotlib as m
import metpy.calc as mpcalc
from metpy.plots import Hodograph, SkewT
from metpy.units import units
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, SkewT
from metpy.units import units
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [14]:
import os
from glob import glob

def process_wrf_file(filepath, lat, lon, time_index=0):
    """
    处理单个WRF文件，计算指定经纬度点的CAPE和CIN
    """
    try:
        # 读取文件
        wrfin = Dataset(filepath)
        # 经纬度转网格坐标
        x, y = ll_to_xy(wrfin, lat, lon)
        # 提取变量（单位需明确指定）
        p = getvar(wrfin, "pressure", timeidx=time_index)[:, x, y] * units.hPa
        T = getvar(wrfin, "tc", timeidx=time_index)[:, x, y] * units.degC
        Td = getvar(wrfin, "td", timeidx=time_index)[:, x, y] * units.degC
        # 计算抬升路径
        parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
        # 计算CAPE和CIN
        cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)
        # 获取时间信息
        all_times = getvar(wrfin, "times")
        current_time = all_times[time_index].values.astype(str)
        wrfin.close()
        return {"time": current_time, "cape": cape.magnitude, "cin": cin.magnitude}
    
    except Exception as e:
        print(f"处理文件 {os.path.basename(filepath)} 时出错: {str(e)}")
        return None


In [15]:
# 定义输入参数
folder_path = r"E:\000_FuJian\wrfout_FuJian\d02"  # 替换为你的文件夹路径
target_lat = 25.5  # 目标纬度
target_lon = 119   # 目标经度
time_index = 0     # 分析的时间步索引

# 获取所有.nc文件
file_list = glob(os.path.join(folder_path, "*.nc"))

# 存储结果
results = []

# 遍历处理每个文件
for file_path in file_list:
    print(f"正在处理文件: {os.path.basename(file_path)}")
    result = process_wrf_file(file_path, target_lat, target_lon, time_index)
    if result:
        results.append(result)

# 转换为DataFrame并保存
df_results = pd.DataFrame(results)
df_results.to_csv("cape_cin_results.csv", index=False)
print("处理完成！结果已保存到 cape_cin_results.csv")


正在处理文件: a1_d02.nc
处理文件 a1_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: a2_d02.nc
处理文件 a2_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b10_d02.nc
处理文件 b10_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b11_d02.nc
处理文件 b11_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b12_d02.nc
处理文件 b12_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b1_d02.nc
处理文件 b1_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b2_d02.nc
处理文件 b2_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b3_d02.nc
处理文件 b3_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b4_d02.nc
处理文件 b4_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b5_d02.nc
处理文件 b5_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b6_d02.nc
处理文件 b6_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b7_d02.nc
处理文件 b7_d02.nc 时出错: 'DataArray' object has no attribute 'to'
正在处理文件: b8_d02.nc
处理文件 b8_d02.nc 时出错: 'DataArr

In [21]:
import numpy as np
import pandas as pd
from netCDF4 import Dataset
from wrf import getvar, ll_to_xy
import metpy.calc as mpcalc
from metpy.units import units

# -----------------------------------------------
# 基础配置
# -----------------------------------------------
wrfin = Dataset(r"E:\000_FuJian\wrfout_FuJian\d02\a2_d02.nc")  # WRF输出文件路径
lat_lon = [25.5, 119]                                          # 目标点经纬度
output_csv = 'hourly_cape_cin.csv'                              # 输出文件名

# -----------------------------------------------
# 核心计算逻辑
# -----------------------------------------------
# 转换地理坐标为网格坐标
x, y = ll_to_xy(wrfin, lat_lon[0], lat_lon[1])

# 初始化结果容器
results = []

# 逐小时处理（假设时间步长=1小时，共25小时）
for time_idx in range(25):
    try:
        # 提取基础变量（压力单位自动转换为hPa）
        p = getvar(wrfin, "pressure", timeidx=time_idx)[:, x, y].metpy.convert_units('hPa')
        temp = getvar(wrfin, "tc",     timeidx=time_idx)[:, x, y] * units.degC
        td = getvar(wrfin, "td",      timeidx=time_idx)[:, x, y] * units.degC
        
        # 手动按气压升序排序（地面→高空）
        sort_idx = np.argsort(-p.magnitude)  # 降序索引
        p_sorted = p[sort_idx]
        t_sorted = temp[sort_idx]
        td_sorted = td[sort_idx]

        # 计算气块路径（使用地面层参数）
        parcel_profile = mpcalc.parcel_profile(p_sorted, t_sorted[0], td_sorted[0])
        
        # 计算CAPE/CIN
        cape, cin = mpcalc.cape_cin(p_sorted, t_sorted, td_sorted, parcel_profile)
        
        # 结果保留两位小数
        results.append({
            'Hour': time_idx,
            'CAPE(J/kg)': round(cape.magnitude, 2),
            'CIN(J/kg)': round(cin.magnitude, 2)
        })
        print(f"成功处理 第{time_idx}小时 | CAPE: {cape.magnitude:.1f} J/kg | CIN: {cin.magnitude:.1f} J/kg")
    
    except Exception as e:
        print(f"错误 @ 第{time_idx}小时: {str(e)}")
        results.append({'Hour': time_idx, 'CAPE(J/kg)': np.nan, 'CIN(J/kg)': np.nan})

# -----------------------------------------------
# 结果保存与展示
# -----------------------------------------------
# 转为DataFrame并保存
result_df = pd.DataFrame(results)
result_df.to_csv(output_csv, index=False, float_format='%.2f')

# 摘要打印
print("\n最终结果摘要:")
print(result_df.dropna().describe())


错误 @ 第0小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第1小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第2小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第3小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第4小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第5小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第6小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第7小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第8小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第9小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第10小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第11小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第12小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第13小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第14小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第15小时: 'DataArray' object has no attribute 'magnitude'
错误 @ 第16小时: 'DataArray' object has no attribute 'm